# Strong and Weak Ties

**Lecture:** NS04 — Strong and Weak Ties  
**Reading:** Easley & Kleinberg, Chapter 3

## Learning objectives

By the end of this notebook you will be able to:

1. Implement and interpret **neighborhood overlap** for each edge in a network.
2. Identify **local bridges** (edges with zero neighborhood overlap) and verify the STCP claim computationally.
3. Reproduce the qualitative finding of Onnela et al. (2007): **overlap increases with tie strength**.
4. Measure **structural holes** per node and identify which nodes bridge otherwise disconnected groups.

In [ ]:
from netsci_utils import *            # nx, np, plt; set_seeds(); setup_matplotlib()
import pandas as pd
from collections import defaultdict

set_seeds()        # RANDOM_SEED = 42
%matplotlib inline

---
## Part 1 — Triadic Closure and Neighborhood Overlap

### 1.1 Theoretical background

> **Definition (NS04):** The *neighborhood overlap* of an edge $(A, B)$ is
> $$O_{AB} = \frac{|N(A) \cap N(B)|}{|N(A) \cup N(B)|}$$
> where $A$ and $B$ themselves are excluded from both sets.

Key facts from the lecture:
- $O_{AB} = 0$ if and only if $(A, B)$ is a **local bridge** (no common neighbors).
- Edges with very small overlap connect nodes from nearly disjoint social circles — they are *almost* local bridges.
- The **Strong Triadic Closure Property (STCP)**: if node $A$ satisfies STCP and has at least two strong ties, then any local bridge $A$ is involved in must be a *weak* tie.

### 1.2 Worked example — small graph

We reproduce the example from the slides: the A–F edge with one common neighbor C.

In [ ]:
def neighborhood_overlap(G, u, v):
    """Neighborhood overlap of edge (u, v).
    O_AB = |N(A) ∩ N(B)| / |N(A) ∪ N(B)|   [NS04]
    Endpoints u and v are excluded from both N(A) and N(B).
    """
    neighbors_u = set(G.neighbors(u)) - {v}
    neighbors_v = set(G.neighbors(v)) - {u}
    common = neighbors_u & neighbors_v
    union  = neighbors_u | neighbors_v
    return len(common) / len(union) if union else 0.0

# Reproduce slide example: A connected to B,C,D,E; F connected to C,G,J
example = nx.Graph()
example.add_edges_from([
    ('A','B'), ('A','C'), ('A','D'), ('A','E'), ('A','F'),
    ('F','C'), ('F','G'), ('F','J')
])

o_AF = neighborhood_overlap(example, 'A', 'F')
print(f"Neighborhood overlap O_AF = {o_AF:.4f}")
print(f"Expected from slides: 1/6 = {1/6:.4f}")
print(f"Is A–F a local bridge? {o_AF == 0}   (span > 2 → local bridge by definition)")

### 1.3 Real network — Zachary's Karate Club

The Karate Club graph (Zachary, 1977) is the canonical social network used throughout the course. 
It has 34 nodes (club members) and 78 edges, and it split into two factions (`Mr. Hi` and `Officer`).

In [ ]:
G = nx.karate_club_graph()
print_network_stats(G)

# Compute overlap for every edge
# O_AB = |N(A) ∩ N(B)| / |N(A) ∪ N(B)|   [NS04]
overlaps = {
    (u, v): neighborhood_overlap(G, u, v)
    for u, v in G.edges()
}

df = pd.DataFrame(
    [(u, v, o) for (u,v), o in overlaps.items()],
    columns=['node_u', 'node_v', 'overlap']
).sort_values('overlap')

print(f"\nOverlap range: [{df.overlap.min():.3f}, {df.overlap.max():.3f}]")
print(f"Mean overlap:  {df.overlap.mean():.3f}")
df.head(10)

### 1.4 Identifying local bridges

In [ ]:
# Local bridges: O_AB = 0  [NS04: "the edge is a local bridge iff neighborhood overlap = 0"]
local_bridges = [(u, v) for (u, v), o in overlaps.items() if o == 0]

print(f"Local bridges: {len(local_bridges)} out of {G.number_of_edges()} edges")
print("\nLocal bridge edges:")
for u, v in local_bridges:
    print(f"  ({u}, {v})  —  spans different parts of the network")

# Visualise: highlight local bridges in red
pos = nx.spring_layout(G, seed=RANDOM_SEED)
clubs = nx.get_node_attributes(G, 'club')
node_colors = ['steelblue' if clubs[n] == 'Mr. Hi' else 'coral' for n in G.nodes()]

regular_edges = [(u, v) for (u, v) in G.edges() if (u, v) not in local_bridges
                                                  and (v, u) not in local_bridges]

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=200, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=regular_edges,
                       edge_color=EDGE_COLOR, width=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=local_bridges,
                       edge_color='red', width=2.5, ax=ax,
                       label='Local bridge')
ax.set_title("Karate Club — local bridges highlighted (red)\nBlue = Mr. Hi faction, Coral = Officer faction")
ax.axis('off')
plt.tight_layout()
plt.show()

**Observation:** Local bridges tend to connect the two factions — consistent with the STCP claim that local bridges are weak ties spanning structurally distant communities.

---
## Part 2 — Tie Strength and Overlap (Onnela et al. 2007)

### 2.1 Theoretical background

> **Key finding (NS04, Onnela et al. 2007):** In the cell-phone communication network, plotting neighborhood overlap as a function of tie-strength percentile shows a clear **positive trend**: stronger ties span denser triangles.

We reproduce this qualitative result on the Karate Club graph, using the number of shared interactions (encoded as edge weight) as a proxy for tie strength.

In [ ]:
# The Karate Club graph includes interaction weights.
# weight = number of common activities between two members.
weights = nx.get_edge_attributes(G, 'weight')

edges_list    = list(overlaps.keys())
overlap_vals  = [overlaps[e] for e in edges_list]
strength_vals = [weights.get(e, weights.get((e[1], e[0]), 1))
                 for e in edges_list]

print(f"Weight range: [{min(strength_vals)}, {max(strength_vals)}]")
print(f"Edges with weight > 1: {sum(w > 1 for w in strength_vals)}")

In [ ]:
# Bin edges into deciles by tie strength, compute mean overlap per bin
n_bins  = 10
percentiles = np.percentile(strength_vals, np.linspace(0, 100, n_bins + 1))

bin_labels, bin_means = [], []
for lo, hi in zip(percentiles[:-1], percentiles[1:]):
    in_bin = [o for s, o in zip(strength_vals, overlap_vals) if lo <= s <= hi]
    if in_bin:
        bin_labels.append(f"{lo:.0f}–{hi:.0f}")
        bin_means.append(np.mean(in_bin))

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.plot(range(len(bin_means)), bin_means, marker='o', linewidth=1.8,
        color=NODE_COLOR)
ax.set_xticks(range(len(bin_labels)))
ax.set_xticklabels(bin_labels, rotation=45, ha='right')
ax.set_xlabel("Tie-strength bin (interaction weight)")
ax.set_ylabel("Mean neighborhood overlap $O_{AB}$")
ax.set_title("Neighborhood overlap increases with tie strength\n"
             "(replicating Onnela et al. 2007, Fig. 2 — qualitative)")
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

# Pearson correlation as a summary
r = np.corrcoef(strength_vals, overlap_vals)[0, 1]
print(f"Pearson r (strength vs overlap) = {r:.3f}")

**Observation:** Even on this small network the positive trend is visible — edges with higher interaction weights (stronger ties) tend to have higher neighborhood overlap, consistent with the cell-phone study findings in NS04.

---
## Part 3 — Structural Holes

### 3.1 Theoretical background

> **Definition (NS04, Burt 2000):** A node that spans multiple local bridges occupies a *structural hole* — the empty space between groups that do not otherwise interact. Such nodes enjoy informational and creative advantages.

A practical proxy: count the number of local-bridge edges incident to each node.

In [ ]:
def local_bridges(G):
    """Return list of all local bridge edges (O_AB = 0)."""
    return [(u, v) for u, v in G.edges()
            if neighborhood_overlap(G, u, v) == 0]

# Count local bridges per node  → structural hole access
lb_edges = local_bridges(G)
hole_count = defaultdict(int)
for u, v in lb_edges:
    hole_count[u] += 1
    hole_count[v] += 1

df_holes = pd.DataFrame(
    [(node, cnt, G.degree(node), clubs.get(node, '?'))
     for node, cnt in hole_count.items()],
    columns=['node', 'structural_holes', 'degree', 'faction']
).sort_values('structural_holes', ascending=False)

print("Top nodes by structural hole access:")
df_holes.head(10)

In [ ]:
# Visualise: node size proportional to structural hole count
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
node_sizes  = [max(hole_count.get(n, 0) * 300, 80) for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.85, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=regular_edges,
                       edge_color=EDGE_COLOR, width=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=lb_edges,
                       edge_color='red', width=2.0, ax=ax)
ax.set_title("Node size ∝ structural hole access\nRed edges = local bridges")
ax.axis('off')
plt.tight_layout()
plt.show()

### 3.2 Exercises

**Exercise 1 (guided):** The `local_bridges()` function above identifies edges with overlap = 0. Now verify the STCP claim computationally:

> *For each local bridge $(u, v)$, check whether it connects nodes from different factions (i.e., is it a cross-faction edge)?*

Count how many local bridges are cross-faction vs. within-faction. Does this support the theory?

In [ ]:
# Exercise 1: count cross-faction vs. within-faction local bridges
# Your solution here
cross_faction_lb = [(u, v) for u, v in lb_edges if clubs[u] != clubs[v]]
within_faction_lb = [(u, v) for u, v in lb_edges if clubs[u] == clubs[v]]

print(f"Local bridges total:          {len(lb_edges)}")
print(f"  Cross-faction (weak ties):  {len(cross_faction_lb)}")
print(f"  Within-faction:             {len(within_faction_lb)}")

**Exercise 2 (open):** Load the `friends.adjlist` dataset from the datasets folder. It does not have weights, so assign synthetic weights using a uniform random draw `weight ~ Uniform(1, 10)`.

1. Reproduce the overlap-vs-strength scatter plot.
2. Is the positive trend still visible?
3. What happens when you shuffle the weights randomly? What does this tell you about the role of network structure vs. weights?

In [ ]:
# Exercise 2: your solution here
SG = load_friends()   # netsci_utils: loads tutorials/datasets/friends.adjlist

# Assign random weights
rng = np.random.default_rng(42)
for u, v in SG.edges():
    SG[u][v]['weight'] = rng.integers(1, 11)

# Your analysis here ...

---
## Summary

Key formulas introduced in this notebook:

| Concept | Formula | Slide reference |
|---|---|---|
| Neighborhood overlap | $O_{AB} = \frac{\|N(A) \cap N(B)\|}{\|N(A) \cup N(B)\|}$ | NS04 |
| Local bridge | $O_{AB} = 0$ | NS04 |
| Structural hole access | $\#\{\text{local bridges incident to node}\}$ | NS04 |

**Key insight:** Weak ties (low-weight edges) are disproportionately local bridges, linking otherwise disconnected communities. Nodes that span structural holes enjoy informational and creative advantages (Burt, 2000).